In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, gc, random, json, time
import numpy as np
import torch

BASE   = "/content/drive/MyDrive/syNNapse_ReID"
DIRS   = ["data","models","embeddings","results","src","notebooks","logs","sample_output"]
for d in DIRS:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)


WORKDIR     = "/content/reid_project"
MODELS_DIR  = f"{BASE}/models"
EMB_DIR     = f"{BASE}/embeddings"
RESULTS_DIR = f"{BASE}/results"
LOGS_DIR    = f"{BASE}/logs"
os.makedirs(WORKDIR, exist_ok=True)

print(f"Project root: {BASE}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/syNNapse_ReID
   GPU: Tesla T4


In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[seed] {SEED} | device: {DEVICE}")


[seed] 42 | device: cuda


In [ ]:
!pip -q install pytorch-metric-learning faiss-cpu kagglehub timm
print("dependencies installed")


dependencies installed


In [ ]:
def _atomic_save(state, dest):
    tmp = f"/content/_tmp_{os.path.basename(dest)}"
    torch.save(state, tmp)
    try:    shutil.move(tmp, dest)
    except: torch.save(state, dest)
    return dest

def save_ckpt(model, optimizer, epoch, loss, prefix, is_best=False):
    state = dict(epoch=epoch, model_state_dict=model.state_dict(),
                 optimizer_state_dict=optimizer.state_dict(), loss=loss)
    tag  = "best" if is_best else f"epoch_{epoch:03d}"
    path = os.path.join(MODELS_DIR, f"{prefix}_{tag}.pth")
    _atomic_save(state, path)
    print(f"  [ckpt] {'BEST ' if is_best else ''}saved → {path}")
    return path

def load_ckpt(path, model, optimizer=None, device="cpu"):
    ckpt  = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", 0) + 1, ckpt

def latest_ckpt(prefix):
    import glob
    files = sorted(glob.glob(os.path.join(MODELS_DIR, f"{prefix}_epoch_*.pth")),
                   key=os.path.getmtime)
    return files[-1] if files else None

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()




In [ ]:
import kagglehub
path      = kagglehub.dataset_download("liucong12601/stanford-online-products-dataset")
DATA_ROOT = os.path.join(path, "Stanford_Online_Products")
print("DATA_ROOT:", DATA_ROOT)
print("Contents:", os.listdir(DATA_ROOT)[:10])


Using Colab cache for faster access to the 'stanford-online-products-dataset' dataset.
DATA_ROOT: /kaggle/input/stanford-online-products-dataset/Stanford_Online_Products
Contents: ['table_final', 'Ebay_info.txt', 'cabinet_final.txt', 'LICENSE', 'bicycle_final', 'mug_final.txt', 'Ebay_test.txt', 'README', 'lamp_final', 'stapler_final']


In [ ]:
import pandas as pd

def parse_sop(txt_file, root):
    rows = []
    with open(txt_file) as f:
        for line in f.readlines()[1:]:
            parts = line.strip().split()
            _, class_id, _, rel_path = parts[0], parts[1], parts[2], parts[3]
            rows.append({"image": os.path.join(root, rel_path), "item_id": int(class_id)})
    return pd.DataFrame(rows)

train_df = parse_sop(os.path.join(DATA_ROOT,"Ebay_train.txt"), DATA_ROOT)
test_df  = parse_sop(os.path.join(DATA_ROOT,"Ebay_test.txt"),  DATA_ROOT)





In [ ]:

unique_ids      = sorted(train_df["item_id"].unique())
id2label        = {iid: i for i, iid in enumerate(unique_ids)}
label2id        = {v: k for k, v in id2label.items()}
train_df["label"] = train_df["item_id"].map(id2label)
NUM_CLASSES     = len(id2label)

In [ ]:

train_df.to_csv(os.path.join(WORKDIR,"train_manifest.csv"), index=False)
test_df.to_csv(os.path.join(WORKDIR,"test_manifest.csv"),  index=False)
with open(os.path.join(WORKDIR,"id2label.json"),"w") as f: json.dump({str(k):v for k,v in id2label.items()}, f)

In [ ]:

print(f"Train  images : {len(train_df):,}  |  items : {NUM_CLASSES:,}")
print(f"Test   images : {len(test_df):,}  |  items : {test_df.item_id.nunique():,}")


Train  images : 59,551  |  items : 11,318
Test   images : 60,502  |  items : 11,316


In [ ]:
from torchvision import transforms

IMG_SIZE   = 224
NORM_MEAN  = [0.485, 0.456, 0.406]
NORM_STD   = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.2)),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
tta_transforms = [
    transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                        transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)]),
    transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)]),
]

print("Transforms defined  (train/test/TTA)")


Transforms defined  (train/test/TTA)


In [ ]:
from torch.utils.data import Dataset, Sampler, DataLoader
from PIL import Image
from collections import defaultdict

class SOPDataset(Dataset):
    #Returns (img_tensor, label_int) for train and (img_tensor, item_id_int, path) for test.
    def __init__(self, df, transform=None, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_train  = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row.image).convert("RGB")
        if self.transform: img = self.transform(img)
        if self.is_train:
            return img, int(row.label)
        return img, int(row.item_id), row.image


class PKBatchSampler(Sampler):
    #Yields batches of P classes × K samples each.
    def __init__(self, labels, P=12, K=4):
        self.P, self.K = P, K
        self.lab2idx   = defaultdict(list)
        for i, lab in enumerate(labels):
            self.lab2idx[int(lab)].append(i)
        self.valid_labs = [l for l, idxs in self.lab2idx.items() if len(idxs) >= K]
        assert len(self.valid_labs) >= P, f"Need ≥P={P} valid classes, got {len(self.valid_labs)}"

    def __len__(self): return len(self.valid_labs) // self.P

    def __iter__(self):
        labs  = self.valid_labs.copy()
        random.shuffle(labs)
        batch = []
        for lab in labs:
            batch.extend(random.sample(self.lab2idx[lab], self.K))
            if len(batch) == self.P * self.K:
                yield batch
                batch = []


P, K = 12, 4

train_ds = SOPDataset(train_df, train_transform, is_train=True)
test_ds  = SOPDataset(test_df,  test_transform,  is_train=False)

train_sampler = PKBatchSampler(train_df["label"].values, P=P, K=K)
train_loader  = DataLoader(train_ds, batch_sampler=train_sampler,
                           num_workers=4, pin_memory=True)
test_loader   = DataLoader(test_ds, batch_size=128, shuffle=False,
                           num_workers=4, pin_memory=True)


imgs, labs = next(iter(train_loader))
u, c = torch.unique(labs, return_counts=True)
print(f"Batch shape: {imgs.shape}  |  unique classes: {len(u)}  |  per-class counts: {c[:6].tolist()}")


Batch shape: torch.Size([48, 3, 224, 224])  |  unique classes: 12  |  per-class counts: [4, 4, 4, 4, 4, 4]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class GeMPooling(nn.Module):
    #Generalised Mean Pooling — outperforms AvgPool for retrieval tasks.
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        # x: (B, C, H, W)
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2), x.size(-1))).pow(1.0/self.p)


class ArcFaceHead(nn.Module):
    #ArcFace margin-based softmax for better class separation during training.
    def __init__(self, in_features, num_classes, s=64.0, m=0.35):
        super().__init__()
        self.s, self.m   = s, m
        self.weight      = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels=None):
        cosine  = F.linear(F.normalize(x), F.normalize(self.weight))   # (B, C)
        if labels is None or not self.training:
            return cosine * self.s
        # add angular margin
        theta   = torch.acos(cosine.clamp(-1+1e-7, 1-1e-7))
        m_mask  = torch.zeros_like(cosine).scatter_(1, labels.view(-1,1), 1.0)
        logits  = torch.cos(theta + self.m * m_mask)
        return logits * self.s


print(" GeMPooling and ArcFaceHead defined")


 GeMPooling and ArcFaceHead defined


In [ ]:
from torchvision import models

class ResNet50Embedder(nn.Module):
    def __init__(self, num_classes, emb_dim=512):
        super().__init__()
        backbone          = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        backbone.avgpool  = GeMPooling(p=3.0)
        self.backbone     = nn.Sequential(*list(backbone.children())[:-1])
        self.proj         = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(2048),
            nn.Linear(2048, emb_dim, bias=False),
            nn.BatchNorm1d(emb_dim),
        )
        self.head = ArcFaceHead(emb_dim, num_classes)

    def forward(self, x, labels=None):
        f   = self.backbone(x)           # (B, 2048, 1, 1)
        emb = self.proj(f)               # (B, 512)
        emb = F.normalize(emb, dim=1)
        logits = self.head(emb, labels)
        return emb, logits

    def get_optimizer(self, lr_head=3e-4, lr_bb=3e-5, wd=1e-4):
        # freeze layers 0-5, unfreeze layer2/3/4 + head
        params = [
            {"params": self.backbone[-1].parameters(), "lr": lr_bb},   # layer4 (index -1 after Sequential)
            {"params": self.proj.parameters(),          "lr": lr_head},
            {"params": self.head.parameters(),          "lr": lr_head},
        ]
        return torch.optim.AdamW(params, weight_decay=wd)



In [ ]:
resnet_model = ResNet50Embedder(NUM_CLASSES, emb_dim=512).to(DEVICE)

n_params = sum(
    p.numel() for p in resnet_model.parameters()
    if p.requires_grad
)

print(f"ResNet50Embedder | trainable params: {n_params/1e6:.1f}M")



ResNet50Embedder | trainable params: 30.4M


In [ ]:
print("Allocated:",
      torch.cuda.memory_allocated()/1024**3,
      "GB")

print("Reserved:",
      torch.cuda.memory_reserved()/1024**3,
      "GB")

Allocated: 0.11602306365966797 GB
Reserved: 0.2578125 GB



    DINOv2 ViT-S/14 with a projection head + ArcFace classifier.
    Uses register tokens + last 4 layers unfrozen for fine-tuning.

In [ ]:
class DINOv2Embedder(nn.Module):
    def __init__(self, num_classes, emb_dim=512, variant="dinov2_vits14"):
        super().__init__()
        self.backbone  = torch.hub.load("facebookresearch/dinov2", variant)
        in_dim         = self.backbone.embed_dim     # 384 for vits14


        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-4:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True

        self.proj = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, emb_dim, bias=False),
            nn.BatchNorm1d(emb_dim),
        )
        self.head = ArcFaceHead(emb_dim, num_classes)

    def forward(self, x, labels=None):
        f      = self.backbone(x)
        emb    = self.proj(f)
        emb    = F.normalize(emb, dim=1)
        logits = self.head(emb, labels)
        return emb, logits

    def get_optimizer(self, lr_head=3e-4, lr_bb=1e-5, wd=1e-4):
        return torch.optim.AdamW([
            {"params": [p for p in self.backbone.parameters() if p.requires_grad], "lr": lr_bb},
            {"params": self.proj.parameters(), "lr": lr_head},
            {"params": self.head.parameters(), "lr": lr_head},
        ], weight_decay=wd)





In [ ]:
dinov2_model = DINOv2Embedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
n_params = sum(p.numel() for p in dinov2_model.parameters() if p.requires_grad)
print(f"DINOv2Embedder  |  trainable params: {n_params/1e6:.1f}M")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:02<00:00, 33.9MB/s]


DINOv2Embedder  |  trainable params: 13.1M


In [ ]:
imgs, labs = next(iter(train_loader))
print(imgs.shape)

torch.Size([48, 3, 224, 224])


In [ ]:
from pytorch_metric_learning.miners import BatchHardMiner
from pytorch_metric_learning.losses import TripletMarginLoss

miner          = BatchHardMiner()
triplet_loss_fn = TripletMarginLoss(margin=0.3)
ce_loss_fn      = nn.CrossEntropyLoss(label_smoothing=0.1)

LAMBDA_TRIPLET = 1.0
LAMBDA_CE      = 0.3

def combined_loss(emb, logits, labels):
    """Batch-hard triplet loss + ArcFace cross-entropy."""
    hard_pairs = miner(emb, labels)
    t_loss     = triplet_loss_fn(emb, labels, hard_pairs)
    ce_loss    = ce_loss_fn(logits, labels)
    return LAMBDA_TRIPLET * t_loss + LAMBDA_CE * ce_loss, t_loss.item(), ce_loss.item()

print(" Combined loss ready  (TripletMargin + ArcFace-CE  λ=1.0/0.3)")


 Combined loss ready  (TripletMargin + ArcFace-CE  λ=1.0/0.3)


In [ ]:
import faiss
from tqdm import tqdm

@torch.no_grad()
def extract_embeddings(model, loader, device):
  #Extract all embeddings, item_ids, and paths from loader.
    model.eval()
    embs, ids, paths = [], [], []
    for batch in tqdm(loader, desc="Extracting embeddings", leave=False):
        imgs, item_ids, img_paths = batch
        imgs = imgs.to(device)
        emb, _ = model(imgs)
        embs.append(emb.cpu().numpy())
        ids.extend([int(x) for x in item_ids])
        paths.extend(list(img_paths))
    return np.vstack(embs).astype("float32"), np.array(ids), np.array(paths)


def topk_accuracy(emb, item_ids, ks=(1,5,10), seed=42, normalize=True):

    # One random image per test item_id = query; rest = gallery.
    # Self-match is excluded from retrieved results.

    if normalize:
        faiss.normalize_L2(emb)

    D      = emb.shape[1]
    index  = faiss.IndexFlatIP(D)
    index.add(emb)

    groups = defaultdict(list)
    for i, iid in enumerate(item_ids):
        groups[iid].append(i)

    rng      = random.Random(seed)
    max_k    = max(ks)
    acc      = {k: 0 for k in ks}
    n_q      = 0
    search_k = min(len(emb), max_k + 1 + 50)

    for iid, idxs in groups.items():
        if len(idxs) < 2: continue
        q_idx      = rng.choice(idxs)
        sims, nbrs = index.search(emb[q_idx:q_idx+1], search_k)
        retrieved  = [j for j in nbrs[0] if j != -1 and j != q_idx]
        ret_ids    = item_ids[retrieved]
        for k in ks:
            if iid in ret_ids[:k]:
                acc[k] += 1
        n_q += 1

    for k in ks: acc[k] = round(acc[k] / max(1, n_q), 4)
    return acc, n_q




In [ ]:

def evaluate_model(model, loader, device, tag=""):
    emb, ids, _ = extract_embeddings(model, loader, device)
    acc, n_q    = topk_accuracy(emb, ids)
    print(f"[{tag}]  Top-1: {acc[1]*100:.2f}%  |  Top-5: {acc[5]*100:.2f}%  |  Top-10: {acc[10]*100:.2f}%  ({n_q} queries)")
    return acc

print("Evaluation functions ready")

Evaluation functions ready


In [ ]:
def train_model(model, train_loader, test_loader, device,
               prefix, epochs=10, eval_every=2, patience=4,
               warmup_epochs=1, lr_head=3e-4, lr_bb=3e-5):


    optimizer = model.get_optimizer(lr_head=lr_head, lr_bb=lr_bb)
    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[lr_bb, lr_head, lr_head],
        total_steps=total_steps, pct_start=0.1, anneal_strategy="cos"
    )

    best_top5     = 0.0
    no_improve    = 0
    history       = {}
    log_path      = os.path.join(LOGS_DIR, f"{prefix}_log.csv")

    with open(log_path, "w") as f:
        f.write("epoch,loss,triplet_loss,ce_loss,top1,top5,top10,lr\n")

    for epoch in range(1, epochs+1):
        model.train()
        epoch_loss = epoch_tl = epoch_ce = 0.0
        t0         = time.time()

        for imgs, labels in tqdm(train_loader, desc=f"[{prefix}] E{epoch}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            emb, logits     = model(imgs, labels)
            loss, tl, cel   = combined_loss(emb, logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            epoch_tl   += tl
            epoch_ce   += cel

        n_batches  = len(train_loader)
        epoch_loss /= n_batches
        epoch_tl   /= n_batches
        epoch_ce   /= n_batches
        lr_now      = scheduler.get_last_lr()[0]

        save_ckpt(model, optimizer, epoch, epoch_loss, prefix)

        acc = {1: 0.0, 5: 0.0, 10: 0.0}
        if epoch % eval_every == 0 or epoch == epochs:
            acc = evaluate_model(model, test_loader, device, tag=f"{prefix} E{epoch}")

        is_best = acc[5] > best_top5
        if is_best:
            best_top5  = acc[5]
            no_improve = 0
            save_ckpt(model, optimizer, epoch, epoch_loss, prefix, is_best=True)
        else:
            no_improve += 1

        history[epoch] = dict(loss=epoch_loss, triplet=epoch_tl, ce=epoch_ce,
                               top1=acc[1], top5=acc[5], top10=acc[10])

        with open(log_path, "a") as f:
            f.write(f"{epoch},{epoch_loss:.4f},{epoch_tl:.4f},{epoch_ce:.4f},"
                    f"{acc[1]:.4f},{acc[5]:.4f},{acc[10]:.4f},{lr_now:.2e}\n")

        elapsed = time.time() - t0
        print(f"  E{epoch:02d}/{epochs}  loss={epoch_loss:.4f}  "
              f"top5={acc[5]*100:.2f}%  lr={lr_now:.1e}  t={elapsed:.0f}s"
              + (" BEST" if is_best else ""))

        clear_gpu()
        if no_improve >= patience:
            print(f"  [early stop] no improvement for {patience} eval cycles")
            break

    print(f"\n Training complete. Best Top-5: {best_top5*100:.2f}%")
    return history, best_top5

print(" train_model() ready")


 train_model() ready



    # Will train all of the three models.
    # Returns: history dict  {epoch: {loss, top1, top5, top10}}

In [ ]:

print("TRAINING ResNet-50 BASELINE")


resnet_model   = ResNet50Embedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
history_resnet, best_resnet = train_model(
    resnet_model, train_loader, test_loader, DEVICE,
    prefix="resnet50", epochs=10, eval_every=2, patience=4,
    lr_head=3e-4, lr_bb=3e-5
)

ckpt_path = os.path.join(MODELS_DIR, "resnet50_best.pth")
if os.path.exists(ckpt_path):
    load_ckpt(ckpt_path, resnet_model, device=DEVICE)

acc_resnet = evaluate_model(resnet_model, test_loader, DEVICE, tag="ResNet50 FINAL")
print(f"ResNet50 → Top-1:{acc_resnet[1]*100:.2f}%  Top-5:{acc_resnet[5]*100:.2f}%  Top-10:{acc_resnet[10]*100:.2f}%")
clear_gpu()


TRAINING ResNet-50 BASELINE


[resnet50] E1:   0%|          | 0/608 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_epoch_001.pth
  E01/10  loss=10.0777  top5=0.00%  lr=3.0e-05  t=379s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_epoch_002.pth


[resnet50 E2]  Top-1: 49.86%  |  Top-5: 62.27%  |  Top-10: 67.15%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_best.pth
  E02/10  loss=9.4245  top5=62.27%  lr=2.9e-05  t=793s BEST


...

[resnet50 E8]  Top-1: 58.23%  |  Top-5: 70.89%  |  Top-10: 75.42%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_best.pth
  E08/10  loss=6.0921  top5=70.89%  lr=3.5e-06  t=801s BEST


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_epoch_009.pth
  E09/10  loss=5.8972  top5=0.00%  lr=9.0e-07  t=377s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/resnet50_epoch_010.pth


[resnet50 E10]  Top-1: 58.08%  |  Top-5: 70.87%  |  Top-10: 75.38%  (11316 queries)
  E10/10  loss=5.8073  top5=70.87%  lr=1.2e-10  t=798s

 Training complete. Best Top-5: 70.89%


[ResNet50 FINAL]  Top-1: 58.23%  |  Top-5: 70.89%  |  Top-10: 75.42%  (11316 queries)
ResNet50 → Top-1:58.23%  Top-5:70.89%  Top-10:75.42%


In [ ]:
del resnet_model
gc.collect()
torch.cuda.empty_cache()

print("Allocated:",
      torch.cuda.memory_allocated()/1024**3, "GB")

Allocated: 0.0166015625 GB


In [ ]:
import timm

class SwinEmbedder(nn.Module):
    def __init__(self, num_classes, emb_dim=512, variant="swin_tiny_patch4_window7_224"):
        super().__init__()
        backbone        = timm.create_model(variant, pretrained=True, num_classes=0)
        in_dim          = backbone.num_features
        self.backbone   = backbone
        self.proj       = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, emb_dim, bias=False),
            nn.BatchNorm1d(emb_dim),
        )
        self.head = ArcFaceHead(emb_dim, num_classes)

    def forward(self, x, labels=None):
        f      = self.backbone(x)
        emb    = self.proj(f)
        emb    = F.normalize(emb, dim=1)
        logits = self.head(emb, labels)
        return emb, logits

    def get_optimizer(self, lr_head=3e-4, lr_bb=3e-5, wd=1e-4):
        return torch.optim.AdamW([
            {"params": self.backbone.parameters(), "lr": lr_bb},
            {"params": self.proj.parameters(),     "lr": lr_head},
            {"params": self.head.parameters(),     "lr": lr_head},
        ], weight_decay=wd)




In [ ]:

swin_model = SwinEmbedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
n_params = sum(p.numel() for p in swin_model.parameters() if p.requires_grad)
print(f"SwinEmbedder  |  trainable params: {n_params/1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SwinEmbedder  |  trainable params: 33.7M


In [ ]:
print("Allocated:",
      torch.cuda.memory_allocated()/1024**3, "GB")

print("Reserved:",
      torch.cuda.memory_reserved()/1024**3, "GB")

Allocated: 0.1436004638671875 GB
Reserved: 0.158203125 GB


In [ ]:

print("TRAINING SWIN TRANSFORMER TINY")


swin_model   = SwinEmbedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
history_swin, best_swin = train_model(
    swin_model, train_loader, test_loader, DEVICE,
    prefix="swin_tiny", epochs=10, eval_every=2, patience=4,
    lr_head=3e-4, lr_bb=2e-5
)

ckpt_path = os.path.join(MODELS_DIR, "swin_tiny_best.pth")
if os.path.exists(ckpt_path):
    load_ckpt(ckpt_path, swin_model, device=DEVICE)

acc_swin = evaluate_model(swin_model, test_loader, DEVICE, tag="SwinTiny FINAL")
print(f"Swin-T → Top-1:{acc_swin[1]*100:.2f}%  Top-5:{acc_swin[5]*100:.2f}%  Top-10:{acc_swin[10]*100:.2f}%")
clear_gpu()


TRAINING SWIN TRANSFORMER TINY


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_epoch_001.pth
  E01/10  loss=9.9730  top5=0.00%  lr=2.0e-05  t=421s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_epoch_002.pth


[swin_tiny E2]  Top-1: 63.70%  |  Top-5: 75.31%  |  Top-10: 78.84%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_best.pth
  E02/10  loss=9.0546  top5=75.31%  lr=1.9e-05  t=881s BEST


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_epoch_003.pth
  E03/10  loss=7.6152  top5=0.00%  lr=1.8e-05  t=426s


...

[swin_tiny E8]  Top-1: 72.36%  |  Top-5: 83.25%  |  Top-10: 86.51%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_best.pth
  E08/10  loss=3.9296  top5=83.25%  lr=2.3e-06  t=884s BEST


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_epoch_009.pth
  E09/10  loss=3.7166  top5=0.00%  lr=6.0e-07  t=427s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_epoch_010.pth


[swin_tiny E10]  Top-1: 72.51%  |  Top-5: 83.32%  |  Top-10: 86.59%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/swin_tiny_best.pth
  E10/10  loss=3.6366  top5=83.32%  lr=8.2e-11  t=867s BEST

 Training complete. Best Top-5: 83.32%


[SwinTiny FINAL]  Top-1: 72.51%  |  Top-5: 83.32%  |  Top-10: 86.59%  (11316 queries)
Swin-T → Top-1:72.51%  Top-5:83.32%  Top-10:86.59%


In [ ]:
del swin_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:

print("Allocated:",
      torch.cuda.memory_allocated()/1024**3, "GB")

Allocated: 0.12645912170410156 GB


DINOv2 ViT-S/14 with a projection head + ArcFace classifier.
Uses register tokens + last 4 layers unfrozen for fine-tuning.

In [ ]:
class DINOv2Embedder(nn.Module):
    def __init__(self, num_classes, emb_dim=512, variant="dinov2_vits14"):
        super().__init__()
        self.backbone  = torch.hub.load("facebookresearch/dinov2", variant)
        in_dim         = self.backbone.embed_dim     # 384 for vits14


        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-4:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True

        self.proj = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, emb_dim, bias=False),
            nn.BatchNorm1d(emb_dim),
        )
        self.head = ArcFaceHead(emb_dim, num_classes)

    def forward(self, x, labels=None):
        f      = self.backbone(x)
        emb    = self.proj(f)
        emb    = F.normalize(emb, dim=1)
        logits = self.head(emb, labels)
        return emb, logits

    def get_optimizer(self, lr_head=3e-4, lr_bb=1e-5, wd=1e-4):
        return torch.optim.AdamW([
            {"params": [p for p in self.backbone.parameters() if p.requires_grad], "lr": lr_bb},
            {"params": self.proj.parameters(), "lr": lr_head},
            {"params": self.head.parameters(), "lr": lr_head},
        ], weight_decay=wd)





In [ ]:
dinov2_model = DINOv2Embedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
n_params = sum(p.numel() for p in dinov2_model.parameters() if p.requires_grad)
print(f"DINOv2Embedder  |  trainable params: {n_params/1e6:.1f}M")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DINOv2Embedder  |  trainable params: 13.1M


In [ ]:

print("TRAINING DINOv2 ViT-S/14  [PRIMARY MODEL]")


dinov2_model   = DINOv2Embedder(NUM_CLASSES, emb_dim=512).to(DEVICE)
history_dino, best_dino = train_model(
    dinov2_model, train_loader, test_loader, DEVICE,
    prefix="dinov2_vits14", epochs=12, eval_every=2, patience=5,
    lr_head=3e-4, lr_bb=1e-5
)

ckpt_path = os.path.join(MODELS_DIR, "dinov2_vits14_best.pth")
if os.path.exists(ckpt_path):
    load_ckpt(ckpt_path, dinov2_model, device=DEVICE)

acc_dino = evaluate_model(dinov2_model, test_loader, DEVICE, tag="DINOv2 FINAL")
print(f"DINOv2 → Top-1:{acc_dino[1]*100:.2f}%  Top-5:{acc_dino[5]*100:.2f}%  Top-10:{acc_dino[10]*100:.2f}%")
clear_gpu()


TRAINING DINOv2 ViT-S/14  [PRIMARY MODEL]


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_epoch_001.pth
  E01/12  loss=9.8394  top5=0.00%  lr=9.4e-06  t=386s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_epoch_002.pth


[dinov2_vits14 E2]  Top-1: 68.47%  |  Top-5: 80.02%  |  Top-10: 83.82%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_best.pth
  E02/12  loss=9.0631  top5=80.02%  lr=9.9e-06  t=815s BEST


...

  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_epoch_004.pth


[dinov2_vits14 E4]  Top-1: 74.04%  |  Top-5: 84.96%  |  Top-10: 88.18%  (11316 queries)
  [ckpt] BEST saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_best.pth
  E04/12  loss=6.3278  top5=84.96%  lr=8.4e-06  t=866s BEST


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_epoch_005.pth
  E05/12  loss=5.3625  top5=0.00%  lr=7.2e-06  t=390s


  [ckpt] saved → /content/drive/MyDrive/syNNapse_ReID/models/dinov2_vits14_epoch_006.pth


Extracting embeddings: 100%|█████████▉| 471/473 [05:21<00:01,  1.29it/s]

In [ ]:
@torch.no_grad()
def extract_embeddings_tta(model, df, device, tta_tfms, batch_size=64):
    #Average embeddings over multiple augmented views.
    model.eval()
    all_emb = []
    all_ids = []
    all_pths= []

    for start in tqdm(range(0, len(df), batch_size), desc="TTA Extraction"):
        batch_df = df.iloc[start:start+batch_size]
        for row in batch_df.itertuples():
            img = Image.open(row.image).convert("RGB")
            views = [tfm(img).unsqueeze(0) for tfm in tta_tfms]
            views = torch.cat(views, dim=0).to(device)   # (n_tta, 3, H, W)
            embs, _ = model(views)
            avg_emb = embs.mean(0, keepdim=True).cpu().numpy()
            all_emb.append(avg_emb)
            all_ids.append(int(row.item_id))
            all_pths.append(row.image)

    return np.vstack(all_emb).astype("float32"), np.array(all_ids), np.array(all_pths)


print("Running DINOv2 + TTA evaluation (this takes a few minutes)...")
emb_tta, ids_tta, pths_tta = extract_embeddings_tta(
    dinov2_model, test_df, DEVICE, tta_transforms, batch_size=64
)
acc_dino_tta, n_q = topk_accuracy(emb_tta, ids_tta)
print(f"DINOv2 + TTA → Top-1:{acc_dino_tta[1]*100:.2f}%  Top-5:{acc_dino_tta[5]*100:.2f}%  Top-10:{acc_dino_tta[10]*100:.2f}%  ({n_q} queries)")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

benchmark = {
    "ResNet-50 (GeM + ArcFace)":    acc_resnet,
    "Swin-Tiny (GeM + ArcFace)":    acc_swin,
    "DINOv2 ViT-S/14 (ArcFace)":   acc_dino,
    "DINOv2 ViT-S/14 + TTA ":     acc_dino_tta,
}

print("\n" + "="*62)
print(f"{'Model':<38} {'Top-1':>7} {'Top-5':>7} {'Top-10':>8}")
print("="*62)
for name, acc in benchmark.items():
    print(f"{name:<38} {acc[1]*100:>6.2f}%  {acc[5]*100:>6.2f}%  {acc[10]*100:>7.2f}%")
print("="*62)

fig, ax = plt.subplots(figsize=(12, 5))
x       = np.arange(len(benchmark))
width   = 0.22
colors  = ["#4C72B0", "#55A868", "#C44E52", "#CCB974"]
names   = list(benchmark.keys())

for i, (k_acc, lbl, col) in enumerate(zip([1,5,10], ["Top-1","Top-5","Top-10"], ["#4C72B0","#55A868","#C44E52"])):
    vals = [benchmark[n][k_acc]*100 for n in names]
    ax.bar(x + (i-1)*width, vals, width, label=lbl, color=col, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=12, ha="right", fontsize=10)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Backbone Benchmark — Inventory Re-ID (Stanford Online Products)")
ax.legend()
ax.set_ylim(0, 100)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "benchmark_comparison.png"), dpi=150)
plt.show()
print("Benchmark chart saved")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
pairs = [
    (history_resnet, "ResNet-50",  "#4C72B0"),
    (history_swin,   "Swin-Tiny",  "#55A868"),
    (history_dino,   "DINOv2 ViT-S/14", "#C44E52"),
]
for ax, (hist, name, col) in zip(axes, pairs):
    epochs = sorted(hist.keys())
    losses = [hist[e]["loss"]   for e in epochs]
    top5s  = [hist[e]["top5"]*100 for e in epochs]
    ax.plot(epochs, losses, "o-", color=col,            label="Loss")
    ax2 = ax.twinx()
    ax2.plot(epochs, top5s,  "s--", color=col, alpha=0.5, label="Top-5%")
    ax.set_title(name); ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss"); ax2.set_ylabel("Top-5 %")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150)
plt.show()


In [ ]:
# Use the TTA embeddings already computed
GALLERY_EMB  = emb_tta
GALLERY_IDS  = ids_tta
GALLERY_REFS = pths_tta

np.save(os.path.join(EMB_DIR, "gallery_embeddings.npy"), GALLERY_EMB)
np.save(os.path.join(EMB_DIR, "gallery_item_ids.npy"),   GALLERY_IDS)
np.save(os.path.join(EMB_DIR, "gallery_refs.npy"),        GALLERY_REFS)

print(f"Gallery saved: {GALLERY_EMB.shape[0]:,} items  |  emb dim: {GALLERY_EMB.shape[1]}")
print(f"  → {EMB_DIR}/gallery_*.npy")
